Import base

In [1]:
import json
import urllib.request
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

Read evidence.json from github

In [2]:
!pip install gdown
!gdown 1JlUzRufknsHzKzvrEjgw8D3n_IRpjzo6

Downloading...
From (original): https://drive.google.com/uc?id=1JlUzRufknsHzKzvrEjgw8D3n_IRpjzo6
From (redirected): https://drive.google.com/uc?id=1JlUzRufknsHzKzvrEjgw8D3n_IRpjzo6&confirm=t&uuid=f5c6e4b6-a011-45af-9af7-7f8034008ffb
To: /content/evidence.json
100% 174M/174M [00:03<00:00, 44.5MB/s]


In [3]:
with open("evidence.json", "r", encoding="utf-8") as f:
    evidence_corpus = json.load(f)

print(len(evidence_corpus))

1208827


In [4]:
first_evidence_id = list(evidence_corpus.keys())[0]

print(first_evidence_id)
print(evidence_corpus[first_evidence_id][:500])

evidence-0
John Bennet Lawes, English entrepreneur and agricultural scientist


Read claims JSON from github

In [5]:
base_url = "https://raw.githubusercontent.com/drcarenhan/COMP90042_2026/refs/heads/main/data"

urls = {
    "train_claims": f"{base_url}/train-claims.json",
    "dev_claims": f"{base_url}/dev-claims.json",
    "test_claims": f"{base_url}/test-claims-unlabelled.json"
}

def load_json_from_url(url):
    with urllib.request.urlopen(url) as f:
        return json.load(f)

train_claims = load_json_from_url(urls["train_claims"])
dev_claims = load_json_from_url(urls["dev_claims"])
test_claims = load_json_from_url(urls["test_claims"])

print(len(train_claims))
print(len(dev_claims))
print(len(test_claims))

1228
154
153


Set up label mapping

In [6]:
label2id = {
    "SUPPORTS": 0,
    "REFUTES": 1,
    "NOT_ENOUGH_INFO": 2,
    "DISPUTED": 3
}

id2label = {v: k for k, v in label2id.items()}

Build training samples

In [7]:
def build_examples(claims):
    examples = []
    for claim_id, item in claims.items():
        claim_text = item["claim_text"]
        evidence_texts = []
        for ev_id in item["evidences"]:
            evidence_texts.append(evidence_corpus[ev_id])
        evidence_text = " ".join(evidence_texts)
        label = label2id[item["claim_label"]]
        examples.append((claim_text, evidence_text, label))
    return examples

train_examples = build_examples(train_claims)
dev_examples = build_examples(dev_claims)

print(len(train_examples))
print(len(dev_examples))
print(train_examples[0])

1228
154
('Not only is there no scientific evidence that CO2 is a pollutant, higher CO2 concentrations actually help ecosystems support more plant and animal life.', 'At very high concentrations (100 times atmospheric concentration, or greater), carbon dioxide can be toxic to animal life, so raising the concentration to 10,000 ppm (1%) or higher for several hours will eliminate pests such as whiteflies and spider mites in a greenhouse. Plants can grow as much as 50 percent faster in concentrations of 1,000 ppm CO 2 when compared with ambient conditions, though this assumes no change in climate and no limitation on other nutrients. Higher carbon dioxide concentrations will favourably affect plant growth and demand for water.', 3)


Load BERT tokenizer

In [8]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Define Dataset class

In [9]:
class FactCheckDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        claim, evidence, label = self.examples[idx]
        encoding = tokenizer(
            claim,
            evidence,
            truncation=True,
            padding="max_length",
            max_length=512,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label)
        }

train_dataset = FactCheckDataset(train_examples)
dev_dataset = FactCheckDataset(dev_examples)

Load BERT classifier

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4,
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Define evaluation standard

In [11]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

Set up training parameters

In [12]:
training_args = TrainingArguments(
    output_dir="./bert_fact_checking",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1"
)

Set up trainer

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics
)

Begin fine tuning

In [14]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,1.015971,0.597403,0.354552
2,No log,0.938614,0.623377,0.445652
3,No log,0.946820,0.603896,0.414630


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=462, training_loss=0.923784132127638, metrics={'train_runtime': 412.9127, 'train_samples_per_second': 8.922, 'train_steps_per_second': 1.119, 'total_flos': 969318533873664.0, 'train_loss': 0.923784132127638, 'epoch': 3.0})

Evaluate dev set

In [15]:
trainer.evaluate()

{'eval_loss': 0.9384710192680359,
 'eval_accuracy': 0.6233766233766234,
 'eval_macro_f1': 0.44565217391304346,
 'eval_runtime': 4.8236,
 'eval_samples_per_second': 31.927,
 'eval_steps_per_second': 4.146,
 'epoch': 3.0}